# Fubon Futures Historical Candles Validation
本 Notebook 只驗證行情資料，不建立交易策略或正式 production warehouse。
開發交付狀態：STATIC TESTED；LIVE HISTORICAL API NOT YET TESTED。

左側 Secrets 設定原本四個 FUBON 名稱。先 Mount Drive；沿用 API Key login。
**預設 MAX_PHASE=1**：只查最近五個完整 regular sessions，重複一次檢查可重現性。
Phase 1 結果檢視後，可設定 MAX_PHASE=7 執行完整小樣本順序；每次皆重新走必要前置 gates。
未達 PASS / PASS WITH CAVEATS 時停止後續 phases；UNRESOLVED 不自動跳過。
branch 尚未 commit/push 時，正式 run 的 clean Git 檢查會停止，不會假冒 provenance。


## 0. Environment / Git / authentication

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from google.colab import userdata
REPO_URL = "https://github.com/hh4832/fubon_futures_intraday_data_validation.git"
REPO_DIR = Path('/content/fubon_futures_intraday_data_validation_historical')
BRANCH = 'feature/historical-candles-validation'
# Public clone first; only a private repo needs optional GITHUB_TOKEN.
if not REPO_DIR.exists():
    result = subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],capture_output=True,text=True)
    if result.returncode:
        try:
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            raise RuntimeError('Public clone failed; private repositories need GITHUB_TOKEN') from None
        if not token:
            raise RuntimeError('GITHUB_TOKEN unavailable')
        with tempfile.TemporaryDirectory() as tmp:
            askpass = Path(tmp)/'askpass.py'
            askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["COLAB_CLONE_TOKEN"])\n')
            askpass.chmod(0o700)
            env = dict(os.environ,GIT_ASKPASS=str(askpass),GIT_TERMINAL_PROMPT='0',COLAB_CLONE_TOKEN=token)
            retry = subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],env=env,capture_output=True)
            del env,token
            if retry.returncode:
                raise RuntimeError('Authenticated clone failed; output suppressed')
remote = subprocess.check_output(['git','remote','get-url','origin'],cwd=REPO_DIR,text=True).strip()
if remote != REPO_URL:
    raise RuntimeError('Repository origin differs from expected URL')
os.chdir(REPO_DIR)
sys.path.insert(0,str(REPO_DIR))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)

actual_branch = subprocess.check_output(['git','branch','--show-current'],cwd=REPO_DIR,text=True).strip()
if actual_branch != BRANCH:
    raise RuntimeError('Use the historical feature branch checkout; no automatic branch replacement')

In [ ]:
from src.archive import git_info
provenance = git_info(REPO_DIR)
print(provenance)
if provenance['git_status']:
    raise RuntimeError('Use a clean Git checkout; no auto pull or overwrite')

In [ ]:
import platform
print({'python':sys.version,'executable':sys.executable,'platform':platform.platform(),'architecture':platform.machine()})
from src.config import load_config
cfg = load_config()
from src.historical_client import load_historical_config
cfg = load_historical_config()
MAX_PHASE = 1  # 1=最小 smoke；2..7=依序擴展，不能跳步
if not 1 <= MAX_PHASE <= 7:
    raise ValueError('MAX_PHASE must be 1..7')
cfg['max_phase'] = MAX_PHASE

In [ ]:
from tqdm.auto import tqdm
from scripts.install_fubon_sdk_colab import install
for _ in tqdm(range(1),desc='SDK setup'):
    sdk_info = install(cfg['sdk_version'],cfg['sdk_download_url'],cfg['sdk_official_sha256'])

In [ ]:
import fubon_neo
from fubon_neo.sdk import FubonSDK
print("SDK import PASS")

In [ ]:
from src.fubon_auth import load_secrets, validate_certificate, login
secrets = load_secrets(userdata.get)

In [ ]:
validate_certificate(secrets)

In [ ]:
sdk, auth_info = login(secrets)
del secrets

In [ ]:
try:
    sdk.init_realtime()
    restfutopt = sdk.marketdata.rest_client.futopt
except Exception:
    raise RuntimeError('Market-data initialization failed; SDK exception suppressed') from None
from src.historical_client import read_canonical_documentation, session_calendar, recent_complete_dates
_, documentation_info = read_canonical_documentation(cfg)
calendar = session_calendar(cfg)
print(documentation_info)
print('Recent completed regular sessions:', recent_complete_dates(calendar,'regular',delay_minutes=cfg['publication_safety_delay_minutes']))
print('Calendar is a scheduled reference with finite reviewed coverage; unexpected closures require separate evidence.')


## 1–7. Ordered validation
1. Recent regular 1! 1m → repeatability
2. Same actual contract / date intraday overlap
3. Concrete contract month
4. Single rollover window
5. Afterhours / same rollover window attribution
6. Historical higher timeframe aggregation by trade_date/session
7. Fixed cross-year samples, including 2017-06

任何 FAIL/UNRESOLVED 阻擋後續；bar count、gaps、average 的不確定性會保留。
canonical historical documentation 讀取原 intraday run archive，SHA256 固定驗證。
所有 response 從第一次請求就直接寫進新的 Drive archive，不覆蓋 canonical output。


In [ ]:
from src.historical_runner import run_historical
run_dir, summary = run_historical(restfutopt, REPO_DIR, sdk_info, auth_info, cfg)
print(summary)


## Diagnostics / archive

In [ ]:
import pandas as pd
for name in ['historical_stages','historical_daily_coverage','historical_schema_validation',
             'historical_vs_intraday_summary','rollover_semantics','historical_aggregation_validation','cross_year_coverage']:
    path = run_dir/'diagnostics'/f'{name}.csv'
    if path.is_file():
        print(name)
        display(pd.read_csv(path))
print('Drive output directory:',str(run_dir.resolve()))


## 結論界線
所有判讀標示 Observed fact / Documented fact / Inference / Unresolved。
Provider historical vs intraday 是 internal consistency，外部驗證獨立標示。
單一窗口不建立 universal rollover 規則，2! / 3! 尚未取樣。
Default Phase 1 即使 PASS，也不是整段 2017–2026 可回測證明。
